# El Niño 2026 — Notebook 04: How did the ocean warm the surface, and what is the human fingerprint?
**le-gis.com climate project**

```
1. STRATIFICATION   →   2. THERMOCLINE        →   3. UPWELLING            →   4. HEAT BUDGET          →   5. FORCING LINK
how layered is the      how sensitive is SST      how strongly is it          what actually warmed        does the ocean change
upper ocean?            to thermocline moves?     brought to the surface?     the sea in 2026?            track human forcing?
```

| Step | Question | Method | Data |
|---|---|---|---|
| 1 | Is the upper equatorial Pacific becoming more layered — by temperature or salt? | Buoyancy frequency N² (TEOS-10), split into thermal and haline parts; trends with Sen's slope + Mann-Kendall (Hamed-Rao); FDR for the section map | GLORYS12 T & S, 2°S–2°N, 1993–2026 |
| 2 | Has the surface become more sensitive to thermocline movements? | Regression of Niño-3 SST′ on D20′ in sliding 15-year windows and per event; effective-sample-size CIs; block bootstrap | GLORYS12 |
| 3 | Weaker upwelling, or warmer upwelled water? | Equatorial and coastal (Bakun) upwelling indices from wind stress; two-term regression SST′ = a·D20′ + b·τx′ | ERA5 monthly, GLORYS12 |
| 4 | Which process warmed Niño-3 and the Peru coast? | Upper-50 m box heat budget (surface flux, zonal/meridional advection, upwelling, residual); sensitivity to layer depth | GLORYS12 faces, ERA5 fluxes |
| 5 | Do the ocean changes track human forcing? (+ Q1 background warming) | GLS regression with AR(1) errors on forcing by agent, controlling for volcanic, solar, ENSO and PDO; **validation checks 1–5** | IGCC 2024 forcing (IPCC AR6 method), NOAA GML gases, NOAA PDO |
| + | Two-way link: does El Niño raise CO₂? | Correlation of the NOAA Mauna Loa CO₂ growth rate with Niño-3.4 (effective sample size) | NOAA GML, ERA5 |

**Honest framing.** Steps 1–4 show *how* the ocean changed and *what* warmed it. Step 5 tests whether the changes *track* human forcing after removing natural factors — strong evidence, not proof. The cause-and-effect test is Notebook 05 (climate models with and without human forcing).

**Runtime (first run):** GLORYS salinity strip + heat-budget faces ≈ 45–70 min; ERA5 ≈ 10–30 min (CDS queue). Everything is cached to Drive.

In [ ]:
!pip -q install gsw pymannkendall statsmodels "cdsapi>=0.7.4" copernicusmarine netCDF4
import os, io, json, time, glob, zipfile, getpass, logging, warnings, threading
import numpy as np, pandas as pd, xarray as xr, matplotlib.pyplot as plt, requests
import gsw, pymannkendall as mk, statsmodels.api as sm
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats as sst
warnings.filterwarnings("ignore"); logging.getLogger("copernicusmarine").setLevel(logging.ERROR)
plt.rcParams.update({"axes.grid": True, "grid.alpha": .25, "font.size": 9})

from google.colab import drive
drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/le-gis_enso"
CACHE, WEB = f"{ROOT}/cache", f"{ROOT}/web"
OUT = f"{WEB}/nb04"; [os.makedirs(d, exist_ok=True) for d in (CACHE, WEB, OUT)]

TODAY = pd.Timestamp.today().normalize(); this_year = TODAY.year
LON0, LON1 = 120, 280
EVENTS = [1997, 2015, 2023, 2026]
COLORS = {1997: "#1f77b4", 2015: "#2ca02c", 2023: "#9467bd", 2026: "crimson"}
R_EARTH, RHO0, CP0 = 6.371e6, 1025.0, 3990.0
SEC_MONTH = 30.4375 * 86400

def clean_time(da):
    da = da.sortby("time"); _, keep = np.unique(da.time.values, return_index=True)
    return da.isel(time=np.sort(keep))

def month_start(t):
    return pd.to_datetime(np.atleast_1d(t)).to_period("M").to_timestamp()

def anom_monthly(s, y0=1993, y1=2020):
    # pandas Series (monthly index) -> anomaly vs y0-y1 calendar-month mean
    base = s[(s.index.year >= y0) & (s.index.year <= y1)]
    clim = base.groupby(base.index.month).mean()
    return s - clim.reindex(s.index.month).values

In [ ]:
# --- Credentials ---
if not os.path.exists(os.path.expanduser("~/.cdsapirc")):
    token = getpass.getpass("CDS Personal Access Token: ")
    open(os.path.expanduser("~/.cdsapirc"), "w").write(f"url: https://cds.climate.copernicus.eu/api\nkey: {token}\n")
import cdsapi, copernicusmarine as cm
cds = cdsapi.Client(quiet=True)
cm.login(username=input("Copernicus Marine username: "),
         password=getpass.getpass("Copernicus Marine password: "), force_overwrite=True)

## Statistical toolbox (used in every step)
| Tool | Why |
|---|---|
| `trend()` — Sen's slope + Mann-Kendall with Hamed-Rao autocorrelation correction | Ocean series are skewed and autocorrelated; ordinary least-squares trends look more significant than they are |
| `neff()` — effective sample size (Bretherton et al. 1999) | 400 monthly values may hold only ~40 independent ones; p-values must use the effective number |
| `corr_test()` — correlation with effective-sample-size p-value | Same reason, for correlations |
| `block_boot()` — moving-block bootstrap | Honest confidence intervals for few events without assuming normality |
| `fdr()` — Benjamini-Hochberg false discovery rate (Wilks 2016) | Testing thousands of grid cells produces false "significant" pixels by chance |
| `phase_surrogates()` — phase-randomised surrogates | Null distributions that keep the autocorrelation of the real data (check 4) |

In [ ]:
def trend(y, per=10):
    # Sen's slope per `per` time steps + Hamed-Rao modified Mann-Kendall p-value
    y = np.asarray(y, float); y = y[np.isfinite(y)]
    if len(y) < 8: return np.nan, np.nan
    r = mk.hamed_rao_modification_test(y)
    return r.slope * per, r.p

def lag1(x):
    x = np.asarray(x, float); x = x[np.isfinite(x)] - np.nanmean(x)
    return np.corrcoef(x[:-1], x[1:])[0, 1] if len(x) > 3 else 0.0

def neff(x, y=None):
    # Bretherton et al. (1999): effective sample size for a mean (y=None) or a correlation
    n = np.isfinite(x).sum() if y is None else (np.isfinite(x) & np.isfinite(y)).sum()
    r1 = lag1(x)
    if y is None: return max(2.0, n * (1 - r1) / (1 + r1))
    r2 = lag1(y); return max(3.0, n * (1 - r1 * r2) / (1 + r1 * r2))

def corr_test(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float); m = np.isfinite(x) & np.isfinite(y)
    r = np.corrcoef(x[m], y[m])[0, 1]; ne = neff(x[m], y[m])
    t = r * np.sqrt((ne - 2) / max(1e-9, 1 - r * r)); p = 2 * sst.t.sf(abs(t), ne - 2)
    return r, p, ne

def block_boot(func, data, n=2000, block=6, seed=0):
    # data: array (n_obs, ...) ; func(sample) -> scalar ; returns 2.5/97.5 percentiles
    rng = np.random.default_rng(seed); N = len(data); nb = int(np.ceil(N / block)); out = []
    for _ in range(n):
        starts = rng.integers(0, max(1, N - block + 1), nb)
        idx = np.concatenate([np.arange(s, s + block) for s in starts])[:N]
        try: out.append(func(data[idx]))
        except Exception: pass
    return np.nanpercentile(out, [2.5, 97.5])

def fdr(p, q=0.05):
    p = np.asarray(p, float); flat = p[np.isfinite(p)]
    if flat.size == 0: return np.zeros_like(p, bool)
    s = np.sort(flat); k = np.where(s <= q * np.arange(1, s.size + 1) / s.size)[0]
    thr = s[k.max()] if k.size else -1
    return np.isfinite(p) & (p <= thr)

def phase_surrogates(y, n=1000, seed=1):
    y = np.asarray(y, float); y = y - y.mean(); F = np.fft.rfft(y); rng = np.random.default_rng(seed); out = []
    for _ in range(n):
        ph = rng.uniform(0, 2 * np.pi, F.size); ph[0] = 0
        if y.size % 2 == 0: ph[-1] = 0
        out.append(np.fft.irfft(np.abs(F) * np.exp(1j * ph), n=y.size))
    return np.array(out)

## Data A — GLORYS12 equatorial strip (temperature: cached in Notebook 02; salinity: new)
2°S–2°N averaged, 120°E–80°W at 1°, 0–400 m, monthly 1993–2026. Same loader as Notebook 02, generalised to any variable.

In [ ]:
ZMAX = 400
GLORYS_MY = "cmems_mod_glo_phy_my_0.083deg_P1M-m"
def anfc_id(var):
    grp = {"thetao": "thetao", "so": "so", "uo": "cur", "vo": "cur"}[var]
    return f"cmems_mod_glo_phy-{grp}_anfc_0.083deg_P1M-m"

def cm_open(ds_id, var, lon_a, lon_b, lat_a, lat_b, z0, z1, t0=None, t1=None):
    kw = dict(dataset_id=ds_id, variables=[var], minimum_longitude=lon_a, maximum_longitude=lon_b,
              minimum_latitude=lat_a, maximum_latitude=lat_b, minimum_depth=z0, maximum_depth=z1)
    if t0 is not None: kw.update(start_datetime=str(t0.date()), end_datetime=str(t1.date()))
    return cm.open_dataset(**kw)[var]

COVER = {}
for key, ds_id in [("my", GLORYS_MY), ("anfc", anfc_id("thetao"))]:
    t = month_start(cm_open(ds_id, "thetao", 150, 151, 0, 1, 0, 1).time.values); COVER[key] = (t.min(), t.max())
print({k: f"{a:%Y-%m} -> {b:%Y-%m}" for k, (a, b) in COVER.items()})

def products_for(months):
    # split a list of month starts into (dataset-key, months) chunks: reanalysis first, then real-time analysis
    out, rest = [], pd.DatetimeIndex(months)
    for key in ("my", "anfc"):
        a, b = COVER[key]; sel = rest[(rest >= a) & (rest <= b)]
        if len(sel): out.append((key, sel)); rest = rest.difference(sel)
    return out

def with_retry(f, tries=3, wait=20):
    for k in range(tries):
        try: return f()
        except Exception as e:
            if k == tries - 1: raise
            print("   retry:", str(e)[:80]); time.sleep(wait)

def strip_year(var, y):
    # 2S-2N mean, 1-degree longitude, 0-400 m, monthly, for one year (cached if final)
    f = f"{CACHE}/glorys_eq_{y}.nc" if var == "thetao" else f"{CACHE}/glorys_eq_{var}_{y}.nc"
    if os.path.exists(f) and y <= this_year - 2:
        da = xr.open_dataarray(f).load()                                       # Notebook 02 cache may carry extra coords (e.g. "source")
        return da.drop_vars([c for c in da.coords if c not in ("time", "lon", "level")]).rename("temp" if var == "thetao" else var)
    months = pd.date_range(f"{y}-01-01", f"{y}-12-01", freq="MS"); months = months[months <= TODAY]
    pieces = []
    for key, sel in products_for(months):
        ds_id = GLORYS_MY if key == "my" else anfc_id(var); t1 = sel[-1] + pd.offsets.MonthEnd(0)
        def get():
            parts = []
            for a, b in [(120, 180), (-180, -80)]:
                da = cm_open(ds_id, var, a, b, -2, 2, 0, ZMAX, sel[0], t1)
                parts.append(da.weighted(np.cos(np.deg2rad(da.latitude))).mean("latitude"))
            da = xr.concat(parts, "longitude")
            da = da.assign_coords(longitude=da.longitude % 360).sortby("longitude").drop_duplicates("longitude")
            n = max(1, int(round(1 / float(np.median(np.diff(da.longitude.values))))))
            da = da.coarsen(longitude=n, boundary="trim").mean().load().rename({"longitude": "lon", "depth": "level"})
            da["time"] = month_start(da.time.values); return da
        pieces.append(with_retry(get))
    da = clean_time(xr.concat(pieces, "time")).rename("temp" if var == "thetao" else var)
    da = da.drop_vars([c for c in da.coords if c not in ("time", "lon", "level")])
    if y <= this_year - 2: da.to_netcdf(f)
    print(f"  {var} {y}: {da.sizes['time']} months"); return da

YEARS = list(range(1993, this_year + 1))
print("GLORYS temperature strip (cached from Notebook 02 for 1993-2024) ...")
T_strip = clean_time(xr.concat([strip_year("thetao", y) for y in YEARS], "time"))
print("GLORYS salinity strip (first run ~1 min per year) ...")
S_strip = clean_time(xr.concat([strip_year("so", y) for y in YEARS], "time"))
T_strip, S_strip = xr.align(T_strip, S_strip, join="inner")
print("Strip:", dict(T_strip.sizes), f"{T_strip.time.values[0]}"[:7], "->", f"{T_strip.time.values[-1]}"[:7])

## Data B — ERA5 from Earth Engine: wind stress, surface heat fluxes, SST (1958–now)
`ECMWF/ERA5/HOURLY` averaged to monthly means on Google's servers (no CDS queue). Area 20°S–15°N, 120°E–70°W at 0.5°.
The `mean_*` bands are already rates (W m⁻², N m⁻²; ECMWF sign convention: downward positive), so no unit conversion is needed.
First run ≈ 30–60 min (12 months fetched in parallel per year); every finished year is cached in Drive.

In [ ]:
import ee
from concurrent.futures import ThreadPoolExecutor
EE_PROJECT = "YOUR-EE-PROJECT"   # <-- same Earth Engine project as Notebooks 02-03
ee.Authenticate(); ee.Initialize(project=EE_PROJECT)

ERA5H = ee.ImageCollection("ECMWF/ERA5/HOURLY")
EE_BANDS = {"mean_eastward_turbulent_surface_stress": "taux", "mean_northward_turbulent_surface_stress": "tauy",
            "mean_surface_net_short_wave_radiation_flux": "sw", "mean_surface_net_long_wave_radiation_flux": "lw",
            "mean_surface_sensible_heat_flux": "sh", "mean_surface_latent_heat_flux": "lh", "sea_surface_temperature": "sst"}
RES = 0.5
LATS = np.round(np.arange(15, -20 - 1e-9, -RES), 2)                              # pixel centres, north -> south
PIECES = [(120.0, 179.5), (-180.0, -70.0)]                                        # pixel centres, split at the dateline
LONS = np.concatenate([np.arange(a, b + 1e-9, RES) for a, b in PIECES]) % 360

def grab(img, a, b):
    w = int(round((b - a) / RES)) + 1
    return ee.data.computePixels({"expression": img, "fileFormat": "NUMPY_NDARRAY",
        "grid": {"dimensions": {"width": w, "height": len(LATS)},
                 "affineTransform": {"scaleX": RES, "shearX": 0, "translateX": a - RES / 2,
                                     "shearY": 0, "scaleY": -RES, "translateY": float(LATS[0]) + RES / 2},
                 "crsCode": "EPSG:4326"}})

def month_fields(t0):
    # monthly mean of all hours; if Earth Engine runs out of memory/time, fall back to 3-hourly, then 6-hourly sampling
    t1 = t0 + pd.offsets.MonthBegin(1)
    for step in (1, 3, 6):
        col = ERA5H.filterDate(str(t0.date()), str(t1.date())).select(list(EE_BANDS))
        if step > 1: col = col.filter(ee.Filter.Or(*[ee.Filter.calendarRange(h, h, "hour") for h in range(0, 24, step)]))
        img = col.mean().unmask(-9999, False).toFloat()
        for k in range(4):
            try:
                parts = [grab(img, a, b) for a, b in PIECES]
                out = {}
                for band, short in EE_BANDS.items():
                    arr = np.concatenate([np.asarray(p[band], dtype=float) for p in parts], axis=1); arr[arr <= -9998] = np.nan
                    out[short] = arr
                return out, step
            except Exception as e:
                msg = str(e).lower()
                if "memory" in msg or "timed out" in msg or "too many pixels" in msg: break           # -> coarser sampling
                time.sleep(10 * (k + 1))                                                            # rate limit / transient
                if k == 3: raise
    raise RuntimeError(f"ERA5 {t0:%Y-%m}: failed even with 6-hourly sampling")

def era5_year(y, last):
    f = f"{CACHE}/era5ee_box_{y}.nc"
    if os.path.exists(f) and y < last.year: return xr.open_dataset(f).load()
    months = [m for m in pd.date_range(f"{y}-01-01", f"{y}-12-01", freq="MS") if m.to_period("M") <= last]
    with ThreadPoolExecutor(max_workers=12) as ex: res = list(ex.map(month_fields, months))
    ds = xr.Dataset({v: (("time", "lat", "lon"), np.stack([r[0][v] for r in res])) for v in EE_BANDS.values()},
                    coords={"time": pd.DatetimeIndex(months), "lat": LATS, "lon": LONS}).sortby("lat")
    ds["sst"] = ds["sst"] - 273.15
    ds.attrs["hour_step"] = str(sorted({r[1] for r in res}))
    if y < last.year: ds.to_netcdf(f)
    return ds

end = pd.Timestamp(ERA5H.filterDate(str((TODAY - pd.DateOffset(days=60)).date()), str((TODAY + pd.DateOffset(days=1)).date()))
                   .aggregate_max("system:time_start").getInfo(), unit="ms")
LAST = (end + pd.Timedelta(hours=1)).to_period("M") - 1                         # last complete month
print("Earth Engine ERA5 available to", end, "-> last complete month", LAST)
parts = []
for y in range(1958, LAST.year + 1):
    t = time.time(); d = era5_year(y, LAST); parts.append(d)
    if time.time() - t > 2: print(f"  {y}: {d.sizes['time']} months, hour step {d.attrs.get('hour_step')}  ({time.time() - t:.0f} s)")
E5 = xr.concat(parts, "time").sortby("time")
E5["qnet"] = E5.sw + E5.lw + E5.sh + E5.lh                                   # downward positive, W m-2
print("ERA5:", list(E5.data_vars), f"{E5.time.values[0]}"[:7], "->", f"{E5.time.values[-1]}"[:7],
      "| tau_x median", round(float(np.nanmedian(E5.taux)), 3), "N/m2 | SST median", round(float(np.nanmedian(E5.sst)), 1), "C")

def box(da, la, lb, lo_a, lo_b):
    d = da.sel(lat=slice(la, lb), lon=slice(lo_a, lo_b))
    return d.weighted(np.cos(np.deg2rad(d.lat))).mean(["lat", "lon"]).to_series()

## Data C — Greenhouse gases and radiative forcing
* **NOAA Global Monitoring Laboratory (GML):** global-mean CO₂, CH₄, N₂O and the Mauna Loa CO₂ growth rate.
* **Indicators of Global Climate Change 2024** (Forster et al., IPCC AR6 method; Zenodo record 15639576): effective radiative forcing (ERF) by agent, 1750–2024.
* **NOAA PDO index** (ERSSTv5): natural Pacific decadal variability, used as a control.

2025–2026 forcing is extended from NOAA concentrations with the standard simplified formulas (CO₂: 5.35·ln(C/C₀); CH₄: 0.036·(√M−√M₀); N₂O: 0.12·(√N−√N₀)); aerosol, volcanic and solar forcing are held at 2024 values. This is stated on the website.

In [ ]:
UA = {"User-Agent": "le-gis-enso/1.0"}
def get_text(url):
    r = requests.get(url, headers=UA, timeout=90); r.raise_for_status(); return r.text

def gml_monthly(gas):
    t = pd.read_csv(io.StringIO(get_text(f"https://gml.noaa.gov/webdata/ccgg/trends/{gas}/{gas}_mm_gl.csv")), comment="#")
    t.columns = [c.strip().lower() for c in t.columns]
    val = next(c for c in t.columns if c in ("average", "monthly average", "value") or c.startswith("average"))
    s = pd.Series(t[val].values, index=pd.to_datetime(dict(year=t.year, month=t.month, day=1)), name=gas)
    return s.where(s > 0)

GHG = pd.concat([gml_monthly(g) for g in ("co2", "ch4", "n2o")], axis=1)
gr = pd.read_csv(io.StringIO(get_text("https://gml.noaa.gov/webdata/ccgg/trends/co2/co2_gr_mlo.csv")), comment="#")
gr.columns = [c.strip().lower() for c in gr.columns]
CO2_GR = pd.Series(gr[gr.columns[1]].values, index=gr["year"].astype(int), name="co2_growth_ppm")
print("NOAA GML:", GHG.dropna(how="all").index[0].date(), "->", GHG.dropna(how="all").index[-1].date(),
      "| CO2 growth rate", CO2_GR.index.min(), "->", CO2_GR.index.max())

# --- ERF by agent (IGCC 2024) ---
ZURL = "https://zenodo.org/records/15639576/files/ClimateIndicator/data-v2025.06.11.zip?download=1"
zpath = f"{CACHE}/igcc_2024.zip"
if not os.path.exists(zpath):
    open(zpath, "wb").write(requests.get(ZURL, headers=UA, timeout=180).content)
zf = zipfile.ZipFile(zpath)
cands = [n for n in zf.namelist() if n.lower().endswith(".csv") and ("erf" in n.lower() or "forcing" in n.lower())]
print("Forcing files in archive:\n ", "\n  ".join(cands[:15]))
pick = sorted(cands, key=lambda n: (("best" not in n.lower()), ("aggreg" not in n.lower()), len(n)))[0]
erf_raw = pd.read_csv(zf.open(pick)); erf_raw.columns = [str(c).strip().lower() for c in erf_raw.columns]
ycol = next(c for c in erf_raw.columns if "year" in c or c in ("time", "unnamed: 0"))
erf_raw = erf_raw.set_index(erf_raw[ycol].astype(float).astype(int)).drop(columns=[ycol])
print("Using:", pick, "| columns:", list(erf_raw.columns))

def cols(*keys, exclude=()):
    return [c for c in erf_raw.columns if any(k in c for k in keys) and not any(x in c for x in exclude)]
c_ghg = cols("co2", "ch4", "n2o", "wmghg", "halogen", "cfc", "f-gas", exclude=("total", "aggreg"))
c_aer = cols("aerosol", "ari", "aci", exclude=("total",))
c_vol, c_sol = cols("volcan"), cols("solar")
c_tot = cols("total")
ERF = pd.DataFrame(index=erf_raw.index)
ERF["ghg"] = erf_raw[c_ghg].sum(axis=1) if c_ghg else np.nan
ERF["aerosol"] = erf_raw[c_aer].sum(axis=1) if c_aer else 0.0
ERF["volcanic"] = erf_raw[c_vol].sum(axis=1) if c_vol else 0.0
ERF["solar"] = erf_raw[c_sol].sum(axis=1) if c_sol else 0.0
ERF["total"] = erf_raw[c_tot[0]] if c_tot else erf_raw.sum(axis=1)
ERF["anthro"] = ERF.total - ERF.volcanic - ERF.solar
print("Mapped columns -> ghg:", c_ghg, "| aerosol:", c_aer, "| volcanic:", c_vol, "| solar:", c_sol, "| total:", c_tot[:1])

# --- extend to the current year from NOAA concentrations ---
ann = GHG.groupby(GHG.index.year).mean()
y_last = int(ERF.index.max())
for y in range(y_last + 1, this_year + 1):
    if y not in ann.index: continue
    c, m_, n_ = ann.loc[y]; c0, m0, n0 = ann.loc[y_last]
    dF = 5.35 * np.log(c / c0) + 0.036 * (np.sqrt(m_) - np.sqrt(m0)) + 0.12 * (np.sqrt(n_) - np.sqrt(n0))
    row = ERF.loc[y_last].copy(); row["ghg"] += dF; row["total"] += dF; row["anthro"] += dF
    ERF.loc[y] = row
ERF = ERF.sort_index(); ERF["extended"] = ERF.index > y_last
print(ERF.tail(6).round(3))

# --- PDO (NOAA NCEI, ERSSTv5) ---
rows = []
for ln in get_text("https://www.ncei.noaa.gov/pub/data/cmb/ersst/v5/index/ersst.v5.pdo.dat").splitlines():
    p = ln.split()
    if len(p) >= 13 and p[0].isdigit() and len(p[0]) == 4:
        rows += [(pd.Timestamp(int(p[0]), m, 1), float(v)) for m, v in enumerate(p[1:13], 1)]
PDO = pd.Series(dict(rows)).sort_index(); PDO = PDO.where(PDO.abs() < 90)
print("PDO:", PDO.dropna().index[0].date(), "->", PDO.dropna().index[-1].date())

## Step 1 — Stratification: buoyancy frequency N², thermal vs haline
N² = g(α ∂Θ/∂z − β ∂S_A/∂z), computed with TEOS-10 (`gsw`). The thermal part (α∂Θ/∂z) and haline part (−β∂S_A/∂z) add up to N².
Regions along the equator: **West** 130–170°E, **Central** 170°E–140°W, **East** 140–90°W.

In [ ]:
def n2_parts(T, S):
    # T: potential temperature, S: practical salinity; dims (time, lon, level). Returns N2, thermal, haline at mid-depths.
    T = T.transpose("time", "lon", "level"); S = S.transpose("time", "lon", "level")
    z = T.level.values; lon = T.lon.values
    p = gsw.p_from_z(-z, 0.0)
    SA = gsw.SA_from_SP(S.values, p[None, None, :], ((lon + 180) % 360 - 180)[None, :, None], 0.0)
    CT = gsw.CT_from_pt(SA, T.values)
    zm = 0.5 * (z[1:] + z[:-1]); pm = 0.5 * (p[1:] + p[:-1]); dz = np.diff(z)          # depth positive down
    SAm, CTm = 0.5 * (SA[..., 1:] + SA[..., :-1]), 0.5 * (CT[..., 1:] + CT[..., :-1])
    a, b = gsw.alpha(SAm, CTm, pm[None, None, :]), gsw.beta(SAm, CTm, pm[None, None, :])
    g = 9.79
    thermal = g * a * (-(CT[..., 1:] - CT[..., :-1]) / dz)                            # z positive upward
    haline = -g * b * (-(SA[..., 1:] - SA[..., :-1]) / dz)
    coords = {"time": T.time, "lon": T.lon, "zmid": zm}
    mk_ = lambda v, n: xr.DataArray(v, dims=("time", "lon", "zmid"), coords=coords, name=n)
    return mk_(thermal + haline, "N2"), mk_(thermal, "thermal"), mk_(haline, "haline")

N2, N2T, N2S = n2_parts(T_strip, S_strip)
def layer_mean(da, z0=0, z1=150):
    d = da.sel(zmid=slice(z0, z1)); w = xr.DataArray(np.gradient(d.zmid.values), dims="zmid", coords={"zmid": d.zmid})
    return d.weighted(w).mean("zmid")
REG = {"West": (130, 170), "Central": (170, 220), "East": (220, 270)}
n2_layer = {k: layer_mean(v) for k, v in {"N2": N2, "thermal": N2T, "haline": N2S}.items()}

def annual(s):
    s = s.dropna(); g = s.groupby(s.index.year)
    return g.mean()[g.count() >= 10]

rows = []
for reg, (a, b) in REG.items():
    for comp in ("N2", "thermal", "haline"):
        s = n2_layer[comp].sel(lon=slice(a, b)).mean("lon").to_series()
        yv = annual(s); slope, p = trend(yv.values, per=10)
        rows.append({"region": reg, "component": comp, "mean_1e-4_s-2": yv.mean() * 1e4,
                     "trend_per_decade_1e-4": slope * 1e4, "trend_%_per_decade": 100 * slope / yv.mean(), "p_value": p})
strat_trends = pd.DataFrame(rows).round(4)
strat_trends.to_csv(f"{OUT}/step1_stratification_trends.csv", index=False)
strat_trends

In [ ]:
# Section map: trend of annual-mean N2 (longitude x depth), FDR-controlled significance
N2a = N2.sel(zmid=slice(0, 300)).groupby("time.year").mean()
N2a = N2a.sel(year=[y for y in N2a.year.values if (N2.time.dt.year == y).sum() >= 10])
slope = np.full(N2a.shape[1:], np.nan); pval = np.full(N2a.shape[1:], np.nan)
for i in range(N2a.sizes["lon"]):
    for j in range(N2a.sizes["zmid"]):
        s, p = trend(N2a.values[:, i, j], per=10); slope[i, j], pval[i, j] = s, p
sig = fdr(pval)
fig, ax = plt.subplots(1, 2, figsize=(16, 4.8), gridspec_kw={"width_ratios": [2.2, 1]})
lim = max(float(np.nanpercentile(np.abs(slope), 98)), 1e-12)
cf = ax[0].contourf(N2a.lon, N2a.zmid, slope.T, levels=np.linspace(-lim, lim, 17), cmap="RdBu_r", extend="both")
yy, xx = np.meshgrid(N2a.zmid, N2a.lon); ax[0].scatter(xx[sig][::3], yy[sig][::3], s=1.5, c="k", alpha=.5)
ax[0].invert_yaxis(); ax[0].set_ylabel("Depth (m)")
ax[0].set_xticks([120, 150, 180, 210, 240, 270]); ax[0].set_xticklabels(["120°E", "150°E", "180°", "150°W", "120°W", "90°W"])
ax[0].set_title(f"Trend of N² per decade, {int(N2a.year[0])}–{int(N2a.year[-1])} (dots: significant after FDR)", loc="left", fontweight="bold")
plt.colorbar(cf, ax=ax[0], label="s⁻² per decade")
st = strat_trends[strat_trends.component != "N2"].pivot(index="region", columns="component", values="trend_per_decade_1e-4").loc[list(REG)]
st.plot.bar(ax=ax[1], color={"thermal": "#dc2626", "haline": "#2563eb"}, rot=0)
ax[1].axhline(0, c="k", lw=.7); ax[1].set_ylabel("trend of 0–150 m N² (10⁻⁴ s⁻² per decade)")
ax[1].set_title("Who drives the change: temperature or salt?", loc="left", fontweight="bold")
plt.tight_layout(); plt.savefig(f"{OUT}/step1_stratification.png", dpi=180, bbox_inches="tight"); plt.show()

## Step 2 — Thermocline: how sensitive is the surface to thermocline movements?
Feedback strength = slope of Niño-3 SST′ on Niño-3 D20′ (°C of surface warming per 10 m of thermocline deepening).
Computed in sliding 15-year windows and, separately, for March–December of each event year.

In [ ]:
def d20(temp):
    T = temp.transpose(..., "level").values; z = temp.level.values
    below = T < 20; k = np.clip(np.argmax(below, axis=-1), 1, len(z) - 1)
    t1 = np.take_along_axis(T, (k - 1)[..., None], -1)[..., 0]; t2 = np.take_along_axis(T, k[..., None], -1)[..., 0]
    out = z[k - 1] + (20 - t1) * (z[k] - z[k - 1]) / (t2 - t1); out[~below.any(axis=-1)] = np.nan
    dims = [d for d in temp.transpose(..., "level").dims if d != "level"]
    return xr.DataArray(out, dims=dims, coords={d: temp[d] for d in dims}, name="d20")

N3 = (210, 270)
D20_n3 = d20(T_strip).sel(lon=slice(*N3)).mean("lon").to_series()
SST_n3 = T_strip.isel(level=0).sel(lon=slice(*N3)).mean("lon").to_series()
tau_n3 = box(E5.taux, -5, 5, *N3)
df2 = pd.DataFrame({"sst": anom_monthly(SST_n3), "d20": anom_monthly(D20_n3), "taux": anom_monthly(tau_n3, 1993, 2020)}).dropna()

def slope_fit(d):
    X = sm.add_constant(d[:, 1]); return sm.OLS(d[:, 0], X).fit().params[1] * 10        # degC per 10 m

WIN = 15 * 12; rows = []
for s in range(0, len(df2) - WIN + 1, 12):
    w = df2.iloc[s:s + WIN]; arr = w[["sst", "d20"]].values
    b = slope_fit(arr); r, p, ne = corr_test(w.sst, w.d20); lo, hi = block_boot(slope_fit, arr, n=800, block=12)
    rows.append({"center": w.index[len(w) // 2].year + .5, "slope": b, "lo": lo, "hi": hi, "r": r, "n_eff": ne})
feedback_win = pd.DataFrame(rows)
ev = {}
for e in EVENTS:
    w = df2[(df2.index >= f"{e}-03-01") & (df2.index <= f"{e}-12-01")]
    if len(w) >= 5:
        arr = w[["sst", "d20"]].values; lo, hi = block_boot(slope_fit, arr, n=1500, block=2)
        ev[e] = {"slope": slope_fit(arr), "lo": lo, "hi": hi, "months": len(w)}
feedback_ev = pd.DataFrame(ev).T
s_tr, p_tr = trend(feedback_win.slope.values[::1], per=1)
print(f"Feedback slope trend across windows: {s_tr:+.3f} °C/10 m per window step (p={p_tr:.2f}, few independent windows!)")
feedback_win.round(3).to_csv(f"{OUT}/step2_feedback_windows.csv", index=False); feedback_ev.round(3).to_csv(f"{OUT}/step2_feedback_events.csv")
feedback_ev.round(3)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4.3), gridspec_kw={"width_ratios": [1.6, 1]})
ax[0].fill_between(feedback_win.center, feedback_win.lo, feedback_win.hi, color="#93c5fd", alpha=.5, label="95% CI (block bootstrap)")
ax[0].plot(feedback_win.center, feedback_win.slope, c="#1d4ed8", lw=2, label="15-year window slope")
ax2 = ax[0].twinx(); e_n2 = annual(n2_layer["N2"].sel(lon=slice(*REG["East"])).mean("lon").to_series()) * 1e4
ax2.plot(e_n2.rolling(15, center=True).mean(), c="#dc2626", lw=1.5, ls="--", label="East-Pacific N² (15-yr mean)")
ax2.set_ylabel("N² 0–150 m (10⁻⁴ s⁻²)", color="#dc2626"); ax[0].set_ylabel("°C SST per 10 m D20")
ax[0].set_title("Thermocline feedback strength vs stratification (Niño-3)", loc="left", fontweight="bold"); ax[0].legend(loc="upper left", fontsize=8)
for i, (e, r) in enumerate(feedback_ev.iterrows()):
    ax[1].errorbar(i, r.slope, yerr=[[r.slope - r.lo], [r.hi - r.slope]], fmt="o", c=COLORS[int(e)], ms=9, capsize=5)
ax[1].set_xticks(range(len(feedback_ev))); ax[1].set_xticklabels([f"{int(e)}" + (" (to date)" if int(e) == this_year else "") for e in feedback_ev.index])
ax[1].set_title("Event slope, Mar–Dec of onset year", loc="left", fontweight="bold"); ax[1].set_ylabel("°C SST per 10 m D20")
plt.tight_layout(); plt.savefig(f"{OUT}/step2_thermocline_feedback.png", dpi=180, bbox_inches="tight"); plt.show()

## Step 3 — Upwelling: weaker upwelling, or warmer upwelled water?
* **Equatorial upwelling index:** −τx over Niño-3 (easterly stress drives Ekman divergence at the equator).
* **Coastal upwelling index (Bakun):** offshore Ekman transport off Peru = τ_alongshore / (ρ|f|), m² s⁻¹ per metre of coast.
* **Separating the two feedbacks:** SST′ = a·D20′ + b·τx′ with standardised variables — *a* is the thermocline feedback, *b* the upwelling (Ekman) feedback.

In [ ]:
UI_eq = -box(E5.taux, -5, 5, *N3)                                                   # N m-2, positive = upwelling-favourable
# Peru coast points ~1 degree offshore (lat, lon_east), alongshore unit vector towards the north-west (equatorward)
pts = [(-6, 278.0), (-8, 279.4), (-10, 280.6), (-12, 281.9), (-14, 283.0), (-15, 283.7)]
ux, uy = -0.50, 0.87
cui = []
for la, lo in pts:
    tx = E5.taux.sel(lat=la, lon=lo, method="nearest"); ty = E5.tauy.sel(lat=la, lon=lo, method="nearest")
    f = 2 * 7.2921e-5 * np.sin(np.deg2rad(la))
    cui.append(((tx * ux + ty * uy) / (RHO0 * abs(f))).to_series())
CUI = pd.concat(cui, axis=1).mean(axis=1)                                          # m2 s-1 per m of coast
SST_coast = box(E5.sst, -10, 0, 270, 280)
up = pd.DataFrame({"UI_eq": UI_eq, "CUI": CUI, "sst_n12": SST_coast}).dropna()

rows = []
for name, s in [("Equatorial upwelling index (N m-2)", up.UI_eq), ("Coastal upwelling index Peru (m2 s-1)", up.CUI)]:
    yv = annual(s); sl, p = trend(yv.values, per=10)
    rows.append({"index": name, "mean": yv.mean(), "trend_per_decade": sl, "trend_%_per_decade": 100 * sl / yv.mean(), "p_value": p,
                 "period": f"{yv.index.min()}-{yv.index.max()}"})
grad = annual(box(E5.sst, -5, 5, 120, 180)) - annual(box(E5.sst, -5, 5, *N3))       # west minus east: the 'thermostat' test
sl, p = trend(grad.values, per=10)
rows.append({"index": "Zonal SST gradient, west minus east (C)", "mean": grad.mean(), "trend_per_decade": sl,
             "trend_%_per_decade": 100 * sl / grad.mean(), "p_value": p, "period": f"{grad.index.min()}-{grad.index.max()}"})
up_trends = pd.DataFrame(rows).round(4); up_trends.to_csv(f"{OUT}/step3_upwelling_trends.csv", index=False)

# two-feedback regression (standardised), whole period and per event (Mar-Dec)
z = (df2 - df2.mean()) / df2.std()
def two_term(d):
    m = sm.OLS(d.sst.values, sm.add_constant(d[["d20", "taux"]].values)).fit(); return m.params[1], m.params[2], m.rsquared
a, b, r2 = two_term(z)
print(f"Whole period 1993-now: thermocline term a = {a:+.2f}, upwelling term b = {b:+.2f}  (standardised), R2 = {r2:.2f}")
ev_rows = []
for e in EVENTS:
    w = z[(z.index >= f"{e}-03-01") & (z.index <= f"{e}-12-01")]
    if len(w) >= 6:
        a_, b_, r2_ = two_term(w); ev_rows.append({"event": e, "a_thermocline": a_, "b_upwelling": b_, "R2": r2_, "months": len(w)})
two_ev = pd.DataFrame(ev_rows).set_index("event").round(2); two_ev.to_csv(f"{OUT}/step3_two_feedbacks_events.csv")
display(up_trends); two_ev

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.2))
for e in [1997, 2015, 2026]:
    for k, (col, lab) in enumerate([("UI_eq", "Equatorial upwelling index anomaly (N m⁻²)"), ("CUI", "Coastal upwelling index anomaly, Peru (m² s⁻¹)")]):
        s = anom_monthly(up[col], 1991, 2020); w = s[(s.index >= f"{e-1}-07-01") & (s.index <= f"{e}-12-01")]
        ax[k].plot(np.arange(len(w)) - 6, w.rolling(3, center=True).mean().values, c=COLORS[e], lw=3 if e == 2026 else 1.6, label=str(e))
        ax[k].set_title(lab, loc="left", fontweight="bold", fontsize=9)
for k in (0, 1):
    ax[k].axhline(0, c="k", lw=.7); ax[k].axvline(0, c="grey", ls=":"); ax[k].set_xlabel("months from January of onset year"); ax[k].legend()
two_ev[["a_thermocline", "b_upwelling"]].plot.bar(ax=ax[2], color=["#7c3aed", "#0ea5e9"], rot=0)
ax[2].axhline(0, c="k", lw=.7); ax[2].set_title("Thermocline vs upwelling feedback (standardised)", loc="left", fontweight="bold", fontsize=9)
plt.tight_layout(); plt.savefig(f"{OUT}/step3_upwelling.png", dpi=180, bbox_inches="tight"); plt.show()

## Step 4 — Heat budget of the upper 50 m: what warmed Niño-3 and the Peru coast?
For a box of depth H: **∂T̄/∂t = Q_surface + zonal advection + meridional advection + vertical (upwelling) + residual**.
Advection uses the flow through each face of the box with temperature relative to the box mean; the upwelling velocity at depth H comes from volume conservation (net horizontal outflow = upwelling through the base).
Penetrating sunlight below H is removed (two-band Paulson & Simpson profile). Sensitivity: H = 40, 50, 60 m gives the error bars.

In [ ]:
BOXES = {"Niño-3": dict(lat=(-5, 5), lon=(210, 270)), "Niño-1+2 (coast)": dict(lat=(-10, 0), lon=(270, 280))}
HMAX, HS = 66, (40, 50, 60)
def lon180(x): return (x + 180) % 360 - 180
DLON = 1 / 12

def cm_period(var, lon_a, lon_b, lat_a, lat_b, z0, z1, reduce=None):
    # whole 1993-now series for a small slab, concatenating reanalysis + real-time analysis
    months = pd.date_range("1993-01-01", TODAY, freq="MS"); parts = []
    for key, sel in products_for(months):
        ds_id = GLORYS_MY if key == "my" else anfc_id(var); t1 = sel[-1] + pd.offsets.MonthEnd(0)
        da = with_retry(lambda: cm_open(ds_id, var, lon180(lon_a), lon180(lon_b), lat_a, lat_b, z0, z1, sel[0], t1))
        da = reduce(da) if reduce else da
        da = da.load(); da["time"] = month_start(da.time.values); parts.append(da)
    return clean_time(xr.concat(parts, "time"))

def box_profile(name):
    f = f"{CACHE}/hb_profile_{name.split()[0]}.nc"
    if os.path.exists(f) and (pd.Timestamp(xr.open_dataarray(f).time.values[-1]) >= TODAY - pd.DateOffset(months=3)):
        return xr.open_dataarray(f).load()
    b = BOXES[name]
    red = lambda da: da.weighted(np.cos(np.deg2rad(da.latitude))).mean(["latitude", "longitude"])
    frac = lambda da: da.isel(depth=0).notnull().mean(["latitude", "longitude"])
    out = []
    for y0 in range(1993, this_year + 1, 4):                                      # 4-year chunks keep memory small
        months = pd.date_range(f"{y0}-01-01", f"{min(y0 + 3, this_year)}-12-01", freq="MS"); months = months[months <= TODAY]
        for key, sel in products_for(months):
            ds_id = GLORYS_MY if key == "my" else anfc_id("thetao"); t1 = sel[-1] + pd.offsets.MonthEnd(0)
            da = with_retry(lambda: cm_open(ds_id, "thetao", lon180(b["lon"][0]), lon180(b["lon"][1]), *b["lat"], 0, HMAX, sel[0], t1))
            prof = red(da).load(); fr = frac(da).load()
            prof = prof.assign_coords(ocean_frac=("time", fr.values)); prof["time"] = month_start(prof.time.values); out.append(prof)
        print(f"   {name}: profile through {months[-1]:%Y}")
    prof = clean_time(xr.concat(out, "time")).rename("T")
    prof.to_netcdf(f); return prof

def face(name, side, var):
    f = f"{CACHE}/hb_face_{name.split()[0]}_{side}_{var}.nc"
    if os.path.exists(f) and (pd.Timestamp(xr.open_dataarray(f).time.values[-1]) >= TODAY - pd.DateOffset(months=3)):
        return xr.open_dataarray(f).load()
    b = BOXES[name]; (la, lb), (lo_a, lo_b) = b["lat"], b["lon"]
    if side == "W": slab = (lo_a, lo_a + DLON, la, lb)
    elif side == "E": slab = (lo_b - DLON, lo_b, la, lb)
    elif side == "S": slab = (lo_a, lo_b, la, la + DLON)
    else: slab = (lo_a, lo_b, lb - DLON, lb)
    da = cm_period(var, *slab, 0, HMAX)
    da = da.isel(longitude=0) if side in "WE" else da.isel(latitude=0)
    da = da.rename(var); da.to_netcdf(f); return da

def edges_weights(z, H):
    e = np.concatenate([[0], 0.5 * (z[1:] + z[:-1]), [z[-1] + (z[-1] - z[-2]) / 2]])
    return np.clip(np.minimum(e[1:], H) - e[:-1], 0, None)                        # thickness of each level inside 0-H

def heat_budget(name, H):
    b = BOXES[name]; (la, lb), (lo_a, lo_b) = b["lat"], b["lon"]
    prof = box_profile(name); z = prof.depth.values; w = edges_weights(z, H)
    Tbar = (prof * xr.DataArray(w, dims="depth", coords={"depth": z})).sum("depth") / w.sum()
    TH = prof.interp(depth=H)
    A = R_EARTH**2 * np.deg2rad(lo_b - lo_a) * (np.sin(np.deg2rad(lb)) - np.sin(np.deg2rad(la))) * prof.ocean_frac
    V = A * H
    def flux(side, vel):
        u = face(name, side, vel); T = face(name, side, "thetao")
        u, T = xr.align(u, T, join="inner"); u, T = u.fillna(0), T.fillna(0)
        zz = u.depth.values; wz = xr.DataArray(edges_weights(zz, H), dims="depth", coords={"depth": zz})
        if side in "WE":
            ds = R_EARTH * np.deg2rad(float(np.median(np.abs(np.diff(u.latitude.values)))))
            ax_ = "latitude"
        else:
            lat0 = float(u.latitude) if u.latitude.size == 1 else float(u.latitude.values[0])
            ds = R_EARTH * np.cos(np.deg2rad(lat0)) * np.deg2rad(float(np.median(np.abs(np.diff(u.longitude.values)))))
            ax_ = "longitude"
        tb = Tbar.reindex(time=u.time)
        vol = (u * wz).sum("depth").sum(ax_) * ds                                     # m3 s-1
        heat = (u * (T - tb) * wz).where(T != 0).sum("depth").sum(ax_) * ds          # degC m3 s-1
        return vol, heat
    Uw, Hw = flux("W", "uo"); Ue, He = flux("E", "uo"); Vs, Hs = flux("S", "vo"); Vn, Hn = flux("N", "vo")
    Wb = (Ue - Uw + Vn - Vs)                                                          # m3 s-1, upward through the base
    t = Wb.time
    sw = box(E5.sw, la, lb, lo_a, lo_b); qn = box(E5.qnet, la, lb, lo_a, lo_b)
    pen = 0.58 * np.exp(-H / 0.35) + 0.42 * np.exp(-H / 23.0)                         # Paulson & Simpson (1977), type I
    q = (qn - pen * sw).reindex(pd.DatetimeIndex(t.values))
    terms = pd.DataFrame(index=pd.DatetimeIndex(t.values))
    k = SEC_MONTH
    terms["surface flux"] = q.values / (RHO0 * CP0 * H) * k
    terms["zonal advection"] = ((Hw - He) / V.reindex(time=t)).values * k
    terms["meridional advection"] = ((Hs - Hn) / V.reindex(time=t)).values * k
    terms["upwelling (vertical)"] = (np.maximum(Wb, 0) * (TH.reindex(time=t) - Tbar.reindex(time=t)) / V.reindex(time=t)).values * k
    Tb = Tbar.reindex(time=t).to_series()
    terms["tendency"] = (Tb.shift(-1) - Tb.shift(1)).values / 2
    terms["residual (mixing etc.)"] = terms["tendency"] - terms[["surface flux", "zonal advection", "meridional advection", "upwelling (vertical)"]].sum(axis=1)
    terms["w_upwelling_m_per_day"] = (Wb / A.reindex(time=t)).values * 86400
    return terms

print("Heat-budget data (first run ~20-40 min: box profiles + 8 face slabs) ...")
HB = {name: {H: heat_budget(name, H) for H in HS} for name in BOXES}
for name in BOXES:
    HB[name][50].round(3).to_csv(f"{OUT}/step4_budget_monthly_{name.split()[0]}.csv")
print("Mean upwelling velocity (m/day):", {n: round(float(HB[n][50]['w_upwelling_m_per_day'].mean()), 2) for n in BOXES})

In [ ]:
# Event attribution: cumulative ANOMALOUS contribution of each term, March -> September of the onset year
TERMS = ["surface flux", "zonal advection", "meridional advection", "upwelling (vertical)", "residual (mixing etc.)"]
TCOL = {"surface flux": "#f59e0b", "zonal advection": "#dc2626", "meridional advection": "#16a34a", "upwelling (vertical)": "#7c3aed", "residual (mixing etc.)": "#94a3b8"}
def event_sums(df, e):
    a = df.copy(); base = a[(a.index.year >= 1993) & (a.index.year <= 2020)]
    a = a - base.groupby(base.index.month).mean().reindex(a.index.month).values
    w = a[(a.index >= f"{e}-03-01") & (a.index <= f"{e}-09-01")]
    return w[TERMS + ["tendency"]].sum()
fig, axs = plt.subplots(1, 2, figsize=(16, 4.6), sharey=False); res = []
for ax, name in zip(axs, BOXES):
    tab = {e: pd.concat([event_sums(HB[name][H], e) for H in HS], axis=1) for e in EVENTS if e >= 1993}
    x = np.arange(len(tab)); bottom_p = np.zeros(len(tab)); bottom_n = np.zeros(len(tab))
    for term in TERMS:
        mid = np.array([tab[e].loc[term, 1] for e in tab]); lo = np.array([tab[e].loc[term].min() for e in tab]); hi = np.array([tab[e].loc[term].max() for e in tab])
        base = np.where(mid >= 0, bottom_p, bottom_n)
        ax.bar(x, mid, bottom=base, color=TCOL[term], label=term, width=.6, edgecolor="white")
        bottom_p += np.where(mid >= 0, mid, 0); bottom_n += np.where(mid < 0, mid, 0)
        for e, m_, l_, h_ in zip(tab, mid, lo, hi): res.append({"box": name, "event": e, "term": term, "degC": m_, "H40-60_min": l_, "H40-60_max": h_})
    ax.scatter(x, [tab[e].loc["tendency", 1] for e in tab], c="k", zorder=5, marker="D", label="observed change")
    ax.axhline(0, c="k", lw=.7); ax.set_xticks(x); ax.set_xticklabels([str(e) + (" (to date)" if e == this_year else "") for e in tab])
    ax.set_ylabel("°C, March → September (anomaly)"); ax.set_title(f"What warmed {name}?", loc="left", fontweight="bold")
h, l = axs[0].get_legend_handles_labels(); fig.legend(h, l, loc="lower center", ncol=6, fontsize=9, frameon=False)
plt.tight_layout(rect=(0, 0.07, 1, 1)); plt.savefig(f"{OUT}/step4_heat_budget.png", dpi=180, bbox_inches="tight"); plt.show()
hb_events = pd.DataFrame(res).round(3); hb_events.to_csv(f"{OUT}/step4_budget_events.csv", index=False)
hb_events.pivot_table(index=["box", "term"], columns="event", values="degC").round(2)

## Step 5 — Forcing link: do the ocean changes track human forcing? (+ Q1 background warming)
Model (annual values): **y = β₀ + β₁·human forcing + β₂·volcanic + β₃·solar + β₄·ENSO + β₅·PDO + ε**, ε ~ AR(1), fitted by GLS (`statsmodels.GLSAR`).

Targets: (a) **tropical-mean SST** (20°S–20N, OISST from Notebook 03) — Q1 background warming; (b) **East-Pacific stratification N²** (Step 1); (c) **zonal SST gradient** (Step 3, 1958–now).

**Validation checks** (all reported in one table): 1 residual diagnostics (Ljung-Box, VIF) · 2 out-of-sample prediction (train first 60 %, test last 40 %) · 3 "any trend" test (AIC vs a straight-line model) · 4 phase-randomised surrogate null (1000×) · 5 robustness (2004–now Argo era; global temperature instead of forcing).

In [ ]:
idx = pd.read_csv(f"{WEB}/indices_monthly_oisst_9120.csv", index_col=0, parse_dates=True)       # from Notebook 03
nino34_ann = annual(box(E5.sst, -5, 5, 190, 240).pipe(lambda s: anom_monthly(s, 1991, 2020)))
pdo_ann = annual(PDO)
F = ERF[["anthro", "volcanic", "solar", "ghg", "aerosol"]].copy()
gmst = None
for n in zf.namelist():
    if n.lower().endswith(".csv") and ("gmst" in n.lower() or "temperature" in n.lower()) and "extreme" not in n.lower():
        g = pd.read_csv(zf.open(n)); g.columns = [str(c).strip().lower() for c in g.columns]
        yc = next((c for c in g.columns if "year" in c or "time" in c), g.columns[0]); vc = [c for c in g.columns if c != yc][0]
        gmst = pd.Series(g[vc].values, index=g[yc].astype(float).astype(int)); print("GMST from", n); break

TARGETS = {
    "Tropical-mean SST (°C)": annual(idx["trop"]),
    "East-Pacific N² 0–150 m (10⁻⁴ s⁻²)": annual(n2_layer["N2"].sel(lon=slice(*REG["East"])).mean("lon").to_series()) * 1e4,
    "Zonal SST gradient W–E (°C)": grad,
}
def design(y, forcing="anthro"):
    human = F[forcing] if isinstance(forcing, str) else forcing                        # name of a forcing column, or a series (e.g. GMST)
    X = pd.DataFrame({"human": human, "volcanic": F.volcanic, "solar": F.solar,
                      "enso": nino34_ann, "pdo": pdo_ann}).reindex(y.index)
    d = pd.concat([y.rename("y"), X], axis=1).dropna(); return d

def fit_glsar(d, cols):
    m = sm.GLSAR(d.y.values, sm.add_constant(d[cols].values), rho=1).iterative_fit(maxiter=20); return m

COLS = ["human", "volcanic", "solar", "enso", "pdo"]; rows, fits = [], {}
for tname, y in TARGETS.items():
    d = design(y);
    if len(d) < 15: print("skip", tname, len(d)); continue
    m = fit_glsar(d, COLS); fits[tname] = (m, d)
    b, se = m.params[1], m.bse[1]; resid = m.resid
    lb_p = float(acorr_ljungbox(resid, lags=[min(3, len(resid) // 5)], return_df=True)["lb_pvalue"].iloc[0])
    Xv = sm.add_constant(d[COLS].values); vif_h = variance_inflation_factor(Xv, 1)
    n_tr = int(len(d) * .6); dtr, dte = d.iloc[:n_tr], d.iloc[n_tr:]
    mtr = fit_glsar(dtr, COLS); pred = mtr.predict(sm.add_constant(dte[COLS].values, has_constant="add"))
    rmse = np.sqrt(np.mean((pred - dte.y.values) ** 2)); rmse_clim = np.sqrt(np.mean((dtr.y.mean() - dte.y.values) ** 2))
    rmse_pers = np.sqrt(np.mean((dtr.y.values[-1] - dte.y.values) ** 2))
    dl = d.copy(); dl["human"] = np.arange(len(dl), dtype=float)
    aic_forcing, aic_line = m.aic, fit_glsar(dl, COLS).aic
    sur = phase_surrogates(d.y.values, n=1000); tnull = []
    for s in sur:
        ds_ = d.copy(); ds_["y"] = s + d.y.mean()
        try: mm = sm.OLS(ds_.y.values, sm.add_constant(ds_[COLS].values)).fit(); tnull.append(mm.tvalues[1])
        except Exception: pass
    t_obs = sm.OLS(d.y.values, sm.add_constant(d[COLS].values)).fit().tvalues[1]
    p_sur = np.mean(np.abs(tnull) >= abs(t_obs))
    d04 = d[d.index >= 2004]; b04 = fit_glsar(d04, COLS).params[1] if len(d04) >= 12 else np.nan
    bg = np.nan
    if gmst is not None:
        dg = design(y, forcing=gmst); bg = fit_glsar(dg, COLS).params[1] if len(dg) >= 15 else np.nan
    rows.append({"target": tname, "years": f"{d.index.min()}-{d.index.max()}", "beta_human_per_Wm2": b, "CI95": f"[{b - 1.96*se:.3f}, {b + 1.96*se:.3f}]",
                 "1_LjungBox_p": lb_p, "1_VIF_human": vif_h, "2_RMSE_test": rmse, "2_RMSE_climatology": rmse_clim, "2_RMSE_persistence": rmse_pers,
                 "3_AIC_forcing": aic_forcing, "3_AIC_straight_line": aic_line, "4_p_surrogate": p_sur,
                 "5_beta_2004_on": b04, "5_beta_with_GMST_instead": bg})
valid = pd.DataFrame(rows).set_index("target"); valid.round(3).to_csv(f"{OUT}/step5_forcing_validation.csv")
valid.round(3).T

In [ ]:
# Q1: forced background warming in 2026 relative to the 1991-2020 baseline (tropical-mean SST model)
F.assign(extended=ERF.extended.reindex(F.index)).to_csv(f"{OUT}/forcing_annual.csv", index_label="year")   # used by Notebook 05
m, d = fits["Tropical-mean SST (°C)"]
b, se = m.params[1], m.bse[1]
dF = F.anthro.loc[this_year] - F.anthro.loc[1991:2020].mean()
print(f"Human forcing {this_year} vs 1991-2020: +{dF:.2f} W/m2 (2025+ extended from NOAA concentrations)")
print(f"Forced tropical-ocean warming in {this_year} vs 1991-2020: {b*dF:+.2f} °C (95% CI {(b-1.96*se)*dF:+.2f} to {(b+1.96*se)*dF:+.2f})")
print(f"Compare: ONI - RONI gap from Notebook 03 = background in the Niño-3.4 raw index")

fig, ax = plt.subplots(1, 2, figsize=(15, 4.2))
ax[0].plot(F.index, F.ghg, c="#dc2626", label="Greenhouse gases"); ax[0].plot(F.index, F.aerosol, c="#2563eb", label="Aerosols (SO₂ etc.)")
ax[0].plot(F.index, F.volcanic, c="#64748b", lw=.8, label="Volcanic"); ax[0].plot(F.index, F.anthro, c="k", lw=2, label="Total human")
ax[0].axvspan(ERF.index[~ERF.extended].max(), F.index.max(), color="#fde68a", alpha=.5, label="extended from NOAA")
ax[0].set_xlim(1950, F.index.max()); ax[0].set_ylabel("W m⁻²"); ax[0].legend(fontsize=8); ax[0].set_title("Radiative forcing by agent (IPCC AR6 method)", loc="left", fontweight="bold")
yy = d.y; fit_y = m.predict(sm.add_constant(d[COLS].values))
ax[1].plot(yy.index, yy, "o-", c="#334155", ms=3, label="observed"); ax[1].plot(yy.index, fit_y, c="#dc2626", label="regression fit")
contrib = b * d.human; ax[1].plot(yy.index, contrib - contrib.loc[1991:2020].mean() + yy.loc[1991:2020].mean(), c="#f59e0b", ls="--", label="human-forced part")
ax[1].legend(fontsize=8); ax[1].set_title("Tropical-mean SST: observed vs forced", loc="left", fontweight="bold")
plt.tight_layout(); plt.savefig(f"{OUT}/step5_forcing.png", dpi=180, bbox_inches="tight"); plt.show()

## Two-way link: does El Niño raise CO₂?
Annual Mauna Loa CO₂ growth rate (NOAA GML) vs Niño-3.4, after removing the slow emissions-driven rise (linear detrend). Tropical drought and fires — including Indonesian peatlands — release extra carbon in El Niño years.

In [ ]:
g = CO2_GR.astype(float)
gd = pd.Series(g.values - np.polyval(np.polyfit(g.index, g.values, 2), g.index), index=g.index)          # remove emissions trend
n34m = anom_monthly(box(E5.sst, -5, 5, 190, 240), 1991, 2020)
best = None
for lag, lab in [(0, "same calendar year"), (1, "previous Jul–Jun (ENSO year)")]:
    if lag == 0: x = annual(n34m)
    else:
        s = n34m.copy(); s.index = s.index + pd.DateOffset(months=6); x = annual(s)                         # Jul(y-1)-Jun(y) -> year y
    j = pd.concat([gd.rename("g"), x.rename("n34")], axis=1).dropna()
    r, p, ne = corr_test(j.n34, j.g); print(f"CO2 growth vs Niño-3.4 ({lab}): r = {r:.2f}, p = {p:.3f}, n_eff = {ne:.0f}, years {j.index.min()}-{j.index.max()}")
    if best is None or abs(r) > abs(best[1]): best = (lab, r, p, j)
lab, r, p, j = best
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(j.n34, j.g, c=j.index, cmap="viridis", s=30)
for yv in (1998, 2016, 2024, 2026):
    if yv in j.index: ax.annotate(str(yv), (j.n34[yv], j.g[yv]), xytext=(4, 4), textcoords="offset points", fontsize=8, fontweight="bold")
k = np.polyfit(j.n34, j.g, 1); xx = np.linspace(j.n34.min(), j.n34.max(), 10); ax.plot(xx, np.polyval(k, xx), c="crimson")
ax.set_xlabel(f"Niño-3.4 anomaly (°C), {lab}"); ax.set_ylabel("CO₂ growth-rate anomaly (ppm/yr)")
ax.set_title(f"El Niño years speed up the CO₂ rise (r = {r:.2f}, p = {p:.3f})", loc="left", fontweight="bold")
plt.tight_layout(); plt.savefig(f"{OUT}/co2_growth_enso.png", dpi=180, bbox_inches="tight"); plt.show()
print(f"Slope: +{k[0]:.2f} ppm/yr of extra CO2 growth per +1 °C Niño-3.4")

In [ ]:
# Summary for the website (numbers filled from this run)
s1 = strat_trends.set_index(["region", "component"])
lines = ["# Notebook 04 summary — ocean physics and the human fingerprint", f"_Run {TODAY:%d %b %Y}. GLORYS12 1993–now, ERA5 1958–now, IGCC 2024 forcing, NOAA GML._", ""]
for reg in REG:
    r_ = s1.loc[(reg, "N2")]; th, ha = s1.loc[(reg, "thermal"), "trend_per_decade_1e-4"], s1.loc[(reg, "haline"), "trend_per_decade_1e-4"]
    lines.append(f"- **Stratification, {reg}:** {r_['trend_%_per_decade']:+.1f} % per decade (p = {r_['p_value']:.2f}); thermal {th:+.3f}, haline {ha:+.3f} ×10⁻⁴ s⁻² per decade.")
for e, r_ in feedback_ev.iterrows():
    lines.append(f"- **Thermocline feedback {int(e)}:** {r_.slope:.2f} °C per 10 m (95% CI {r_.lo:.2f}–{r_.hi:.2f}).")
for _, r_ in up_trends.iterrows():
    lines.append(f"- **{r_['index']}:** trend {r_['trend_%_per_decade']:+.1f} % per decade (p = {r_['p_value']:.2f}), {r_['period']}.")
for name in BOXES:
    t = hb_events[(hb_events.box == name) & (hb_events.event == this_year)].set_index("term").degC
    if len(t): lines.append(f"- **Heat budget {name}, Mar–Sep {this_year}:** " + ", ".join(f"{k} {v:+.1f} °C" for k, v in t.items()))
for tname, r_ in valid.iterrows():
    lines.append(f"- **Forcing link — {tname}:** β = {r_.beta_human_per_Wm2:.3f} per W m⁻² {r_.CI95}; surrogate p = {r_['4_p_surrogate']:.3f}; "
                 f"out-of-sample RMSE {r_['2_RMSE_test']:.3f} vs climatology {r_['2_RMSE_climatology']:.3f}; AIC forcing {r_['3_AIC_forcing']:.1f} vs straight line {r_['3_AIC_straight_line']:.1f}.")
lines.append(f"- **El Niño → CO₂:** r = {r:.2f} (p = {p:.3f}); +{k[0]:.2f} ppm/yr per +1 °C Niño-3.4.")
open(f"{OUT}/summary04.md", "w").write("\n".join(lines)); print("\n".join(lines)); print("\nSaved to", OUT)

## Conclusion — what caused El Niño 2026, and is climate change behind it?

**1. The event was made by the ocean–wind system itself (Notebooks 02–03).** Heat stored below the surface during the La Niña years was released by two waves of westerly wind bursts (January and March–April 2026). The Kelvin waves they launched reached South America in late March and May–June, and the Bjerknes feedback locked the event in from June.

**2. The warming came from the ocean, not from the air (Step 4, refined in Notebook 06).** The mixed-layer budget with feedback decomposition (Notebook 06, 0–50 m, 84–95 % closure in Niño-3) shows that Niño-3 warmed by +3.7 °C from March to September 2026 mainly through an exceptionally strong **zonal advective feedback (+2.6 °C**, against +1.4, +0.5 and +0.1 °C in 1997, 2015 and 2023; confirmed by satellite currents: +2.4 °C) and a **thermocline feedback of +2.6 °C** (same as 1997). The Ekman (upwelling) feedback cooled (−0.6 °C), and surface fluxes removed 2.5 °C. Near the South American coast the budget does not close at 0.5° resolution (residual 80–240 % of the change within ~4° of the coast), so the coast is described with wind-based upwelling indices and satellite data. The simple box budget above is kept for reference only.

**3. A thermocline-driven event, with normal-strength feedback (Steps 2–3).** The thermocline term outweighs the upwelling-wind term in all four events (2026: 1.44 vs 0.38, standardised). The thermocline feedback in 2026 (1.16 °C per 10 m) is the same as in 1997, 2015 and 2023 (1.22–1.33). 2026 was strong because the thermocline anomaly was large, not because the feedback had become stronger. Upwelling-favourable winds show no significant trend since 1958, either along the equator (+0.9 % per decade, p = 0.14) or off Peru (+1.1 % per decade, p = 0.30).

**4. Has the ocean changed in a greenhouse-like way? Partly (Steps 1, 3).**
* West and central Pacific: 0–150 m stratification fell 6–8 % per decade since 1993, almost entirely through temperature. The thermocline has deepened (the strongest gradient moved below 150 m), as expected from the stronger trade winds of recent decades.
* East Pacific: stratification rose only slightly (+1.5 % per decade, p = 0.06), and through salinity, not temperature. The thermal stratification increase that climate models project for greenhouse warming (Cai et al. 2018) is **not yet visible** in 1993–2025.
* The west–east temperature contrast strengthened (+1.8 % per decade, p = 0.03): a La Niña-like trend, the opposite of what most climate models project (the "Pacific pattern puzzle", Seager et al. 2019).

**5. The human fingerprint is in the background, not in the El Niño mechanism (Step 5).** Tropical-mean sea temperature tracks human forcing: β = 0.34 °C per W m⁻², surrogate p < 0.001, and good out-of-sample prediction. That puts about +0.37 °C (95 % CI +0.28 to +0.46 °C) of human-forced warming under the 2026 event relative to 1991–2020 (human forcing +1.10 W m⁻²; re-printed below), about two-thirds of the 0.57 °C background in the raw Niño-3.4 index. It raises every raw index and helped the coast reach a record +4.7 °C. Two cautions apply. The forcing model is only marginally better than a straight line (AIC −60.7 vs −59.4). And a perfect-model test with climate models (Notebook 05, only two usable models) showed this regression's textbook p-values to be over-confident. The attribution of the long-term warming to human forcing therefore rests on IPCC AR6, which states it unequivocally, while our regression is *consistent* with it.

**6. Did climate change make El Niño 2026 stronger or more likely? Not demonstrable with these data.** Nothing in the feedback, upwelling or stratification measurements shows a forced change in El Niño itself. This matches IPCC AR6, which has low confidence in any human-caused change in observed ENSO so far. Climate models project more frequent extreme El Niño events and stronger rainfall swings as warming continues (Cai et al. 2018, 2021; IPCC AR6). Our model-based attribution (Notebook 05) could not answer this question robustly, because the public CMIP6 cloud archive held too few suitable runs.

**7. A loop back to the atmosphere.** El Niño years speed up the CO₂ rise: r = 0.52 (p < 0.001), +0.32 ppm/yr per +1 °C Niño-3.4 (1998, 2016 and 2024 stand out), through drought and fire in the tropics, Indonesia included. The 2026–27 event should produce a faster-than-usual CO₂ rise next year.

**In one sentence:** El Niño 2026 was triggered and grown by natural ocean–atmosphere dynamics acting on a large store of subsurface heat, and it played out on a tropical ocean already warmed, consistent with human forcing; that warming made the event hotter in absolute terms, but these data cannot show that it changed El Niño itself.

In [ ]:
# numbers quoted in the conclusion (re-run after updating the data)
m, d = fits["Tropical-mean SST (°C)"]; b, se = m.params[1], m.bse[1]
dF = F.anthro.loc[this_year] - F.anthro.loc[1991:2020].mean()
print(f"Human forcing {this_year} vs 1991-2020: +{dF:.2f} W/m2 -> forced tropical-ocean warming {b*dF:+.2f} °C (95% CI {(b-1.96*se)*dF:+.2f} to {(b+1.96*se)*dF:+.2f})")
for tname, (mm, dd) in fits.items():
    print(f"{tname}: beta {mm.params[1]:.3f}, AIC forcing {valid.loc[tname, '3_AIC_forcing'] if tname in valid.index else float('nan'):.1f} "
          f"vs straight line {valid.loc[tname, '3_AIC_straight_line'] if tname in valid.index else float('nan'):.1f}")

## Website export — data for the interactive 3D ocean (`web/ocean3d.json`)
Monthly Jul 2025 → latest, plus the 1993–2020 climatology for each calendar month ("normal year"):
equatorial temperature section 0–300 m (GLORYS, 2°S–2°N, 1°), surface SST 15°S–15°N (ERA5, 2°×1°), 20 °C thermocline depth,
equatorial zonal wind stress (ERA5) and box-mean upwelling speed from the heat budget (Step 4). Temperatures are packed as base64 int16 (°C × 100).

In [ ]:
import base64
def pack(a):
    a = np.asarray(a, dtype=float); q = np.where(np.isfinite(a), np.round(a * 100), -32768).astype("<i2")
    return base64.b64encode(q.tobytes()).decode()
M0 = pd.Timestamp("2025-07-01")
tm = pd.DatetimeIndex(T_strip.time.values); em = pd.DatetimeIndex(E5.time.values)
MONTHS = pd.DatetimeIndex([t for t in tm if t >= M0 and t in em])
DEP = np.arange(0, 301, 10.0); LON_S = np.arange(120, 281, 1.0)
LON_F = np.arange(120, 281, 2.0); LAT_F = np.arange(-15, 16, 1.0); LON_T = np.arange(130, 271, 10.0)
lev = np.maximum(DEP, float(T_strip.level.min()))
sec = T_strip.interp(level=lev, lon=LON_S).transpose("time", "level", "lon")
sec_m = sec.sel(time=MONTHS); sec_c = sec.sel(time=slice("1993", "2020")).groupby("time.month").mean().transpose("month", "level", "lon")
d20_full = d20(T_strip.interp(lon=LON_S))
d20_m = d20_full.sel(time=MONTHS); d20_c = d20_full.sel(time=slice("1993", "2020")).groupby("time.month").mean()
sfc = E5.sst.interp(lat=LAT_F, lon=LON_F).transpose("time", "lat", "lon")
sfc_m = sfc.sel(time=MONTHS); sfc_c = sfc.sel(time=slice("1993", "2020")).groupby("time.month").mean().transpose("month", "lat", "lon")
tx = E5.taux.sel(lat=slice(-2, 2)).mean("lat").interp(lon=LON_T)
tx_m = tx.sel(time=MONTHS); tx_c = tx.sel(time=slice("1993", "2020")).groupby("time.month").mean()
W = {k: HB[name][50]["w_upwelling_m_per_day"] for k, name in (("n3", "Niño-3"), ("n12", "Niño-1+2 (coast)"))}
w_m = {k: [None if not np.isfinite(v) else round(float(v), 2) for v in s.reindex(MONTHS).values] for k, s in W.items()}
w_c = {k: [round(float(v), 2) for v in s[(s.index.year >= 1993) & (s.index.year <= 2020)].groupby(lambda t: t.month).mean().reindex(range(1, 13)).values] for k, s in W.items()}
r2 = lambda a: [None if not np.isfinite(v) else round(float(v), 1) for v in np.asarray(a).ravel()]
r4 = lambda a: [None if not np.isfinite(v) else round(float(v), 4) for v in np.asarray(a).ravel()]
O3 = {"version": 1, "base": "1993-2020", "updated": f"{TODAY:%Y-%m-%d}",
      "source": "GLORYS12 (Copernicus Marine) temperature; ERA5 (ECMWF via Earth Engine) SST and wind stress; heat budget from Notebook 04",
      "months": [f"{t:%Y-%m}" for t in MONTHS],
      "depth": DEP.tolist(), "lon_sec": LON_S.tolist(), "lon_sfc": LON_F.tolist(), "lat_sfc": LAT_F.tolist(), "lon_tau": LON_T.tolist(),
      "sec": pack(sec_m.values), "clim_sec": pack(sec_c.values), "sfc": pack(sfc_m.values), "clim_sfc": pack(sfc_c.values),
      "d20": [r2(r) for r in d20_m.values], "clim_d20": [r2(r) for r in d20_c.values],
      "taux": [r4(r) for r in tx_m.values], "clim_taux": [r4(r) for r in tx_c.values], "w": w_m, "clim_w": w_c}
json.dump(O3, open(f"{WEB}/ocean3d.json", "w"), separators=(",", ":"))
print(f"ocean3d.json: {len(MONTHS)} months ({O3['months'][0]} -> {O3['months'][-1]}), {os.path.getsize(f'{WEB}/ocean3d.json') / 1e6:.2f} MB")
print("Niño-3 upwelling (m/day):", dict(zip(O3["months"], w_m["n3"])), "| normal:", w_c["n3"])

## Reading the results
* **Step 1:** a positive N² trend in the East with a dominant **thermal** part is the greenhouse fingerprint expected by Cai et al. (2018); a dominant **haline** part points to rainfall/freshwater changes.
* **Step 2–3:** if the 2026 event slope (a) is high while the upwelling term (b) is small, 2026 was a **thermocline-driven** event — warm water was upwelled, not less water.
* **Step 4:** the stacked bars say which process supplied the warming; a large residual flags mixing or data uncertainty.
* **Step 5:** trust the forcing link only if checks pass: Ljung-Box p > 0.05, VIF < 5, out-of-sample RMSE below climatology and persistence, AIC(forcing) clearly lower than AIC(straight line), surrogate p < 0.05, and similar β from 2004 on and with GMST.
* **Cause and effect:** Notebook 05 (CMIP6 runs with vs without human forcing).

**References:** Bjerknes 1969; Paulson & Simpson 1977 *JPO*; Bakun 1990 *Science*; Clement et al. 1996 *J. Climate*; Bretherton et al. 1999 *J. Climate*; Hamed & Rao 1998 *J. Hydrol.*; Page et al. 2002 *Nature*; McDougall & Barker 2011 (TEOS-10/GSW); Wilks 2016 *BAMS*; Santoso et al. 2017 *Rev. Geophys.*; Cai et al. 2018 *Nature*; Seager et al. 2019 *Nat. Clim. Change*; Cai et al. 2023 *Nat. Rev. Earth Environ.*; Forster et al. 2025 *ESSD* (Indicators of Global Climate Change 2024); Hersbach et al. 2020 (ERA5); Lellouche et al. 2021 (GLORYS12).